# Train the pdf2takeoff symbol detector (YOLOX-nano, Apache-2.0 stack)

Trains the tier-2 Scan detector on free Kaggle GPUs (enable **Settings → Accelerator → GPU T4 x2**).
Everything in the stack is Apache-2.0: mmdetection + mmengine + YOLOX — deliberately **not**
Ultralytics YOLO (AGPL), so the model has no license contamination.

**Inputs** — attach a Kaggle Dataset (Studio → New Dataset → upload) produced by either
repo generator, both emit the same layout:

| Source | Command | Notes |
| --- | --- | --- |
| Synthetic sheets | `node scripts/generate-synthetic-dataset.mjs --count 2000 --seed 42` | pretraining, unlimited |
| FloorPlanCAD | `python3 scripts/floorplancad-convert.py --src <dxf> --out training/floorplancad` | real-CAD transfer |

**Outputs** — `symbols.onnx` + `symbols.json` ready to drop into the repo's `public/models/`
(fully gitignored): the browser Scan auto-upgrades to the detector on next load.

> Privacy rule: only public/synthetic data goes to Kaggle. Client drawings are fine-tuned
> on owned hardware only.

In [ ]:
# --- configuration ---------------------------------------------------------
DATASET_DIR = "/kaggle/input/pdf2takeoff-symbols"   # your attached dataset (has dataset.yaml)
CLASSES = ["downlight", "gpo", "switch", "smoke-alarm", "diffuser"]
INPUT_SIZE = 416      # YOLOX-nano native size; must match symbols.json later
MAX_EPOCHS = 100
BATCH_SIZE = 16       # drop to 8 if a T4 OOMs
WORK_DIR = "/kaggle/working/run"

import os, pathlib
assert pathlib.Path(DATASET_DIR).is_dir(), f"attach the dataset and set DATASET_DIR: {DATASET_DIR}"
for split in ("train", "val"):
    imgs = list((pathlib.Path(DATASET_DIR) / "images" / split).glob("*.png"))
    lbls = list((pathlib.Path(DATASET_DIR) / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

In [ ]:
# --- install (match the wheel to Kaggle's preinstalled torch) --------------
import torch, torch.version.cuda
print("torch", torch.__version__, "cuda", torch.version.cuda)
# mmcv prebuilt wheel index pattern: https://download.openmmlab.com/mmcv/dist/<cuda>/<torch>/index.html
# e.g. torch 2.4 + cu121:
!pip install -q mmengine
!pip install -q mmcv --find-links https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!pip install -q "mmdet>=3.3"
import mmengine, mmdet
print("mmengine", mmengine.__version__, "| mmdet", mmdet.__version__)

In [ ]:
# --- sanity: look at the data before burning GPU hours ---------------------
import random, matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def show_sample(split="train", n=4):
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4 * n))
    imgs = sorted((pathlib.Path(DATASET_DIR) / "images" / split).glob("*.png"))
    for ax, p in zip(axes, random.Random(0).sample(imgs, min(n, len(imgs)))):
        lbl = pathlib.Path(DATASET_DIR) / "labels" / split / (p.stem + ".txt")
        im = Image.open(p).convert("RGB"); d = ImageDraw.Draw(im); W, H = im.size
        for line in lbl.read_text().splitlines():
            c, cx, cy, w, h = line.split()
            d.rectangle([(float(cx)-float(w)/2)*W, (float(cy)-float(h)/2)*H,
                         (float(cx)+float(w)/2)*W, (float(cy)+float(h)/2)*H],
                        outline="red", width=3)
        ax.imshow(im); ax.set_title(p.name, fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

show_sample()

In [ ]:
# --- YOLOX-nano config (nano = depth 0.33 / width 0.25 of YOLOX-S) ----------
from mmengine.config import Config

cfg = Config.fromfile("mmdet::yolox/yolox_s_8xb8-300e_coco")

# shrink the backbone to nano
nf = cfg.model.bbox_head
cfg.model.bbox_head.update(num_classes=len(CLASSES))
cfg.model.backbone.deepen_factor, cfg.model.backbone.widen_factor = 0.33, 0.25
nf.in_channels = [256, 512, 1024]        # width-25% of S
cfg.model.bbox_head.feat_channels = 96   # 384 * 0.25
cfg.model.bbox_head.stacked_convs = 2
cfg.model.test_cfg.score_thr = 0.25
cfg.model.test_cfg.nms.iou_threshold = 0.45

# data pipelines at our input size, YOLOX-style letterbox + HSV
train_tf = [
    dict(type="Mosaic", img_scale=(INPUT_SIZE, INPUT_SIZE), pad_val=114.0),
    dict(type="RandomAffine", scaling_ratio_range=(0.1, 2.0), border=(-INPUT_SIZE // 2, -INPUT_SIZE // 2)),
    dict(type="YOLOXHSVRandomAug"),
    dict(type="RandomFlip", prob=0.5),
    dict(type="Resize", img_scale=(INPUT_SIZE, INPUT_SIZE), keep_ratio=True),
    dict(type="Pad", size_divisor=32, pad_val=(114, 114, 114)),
    dict(type="PackDetInputs"),
]
test_tf = [
    dict(type="Resize", img_scale=(INPUT_SIZE, INPUT_SIZE), keep_ratio=True),
    dict(type="Pad", size_divisor=32, pad_val=(114, 114, 114)),
    dict(type="PackDetInputs", meta_keys=("img_id", "img_path", "ori_shape", "img_shape", "scale_factor")),
]
cfg.train_pipeline = train_tf
cfg.test_pipeline = test_tf
cfg.train_dataloader.dataset.update(
    data_prefix=dict(img="images/train/"),
    ann_file="labels/train/", pipeline=train_tf)
cfg.val_dataloader.dataset.update(data_prefix=dict(img="images/val/"), ann_file="labels/val/", pipeline=test_tf)
cfg.test_dataloader = cfg.val_dataloader.deepcopy()

for name in ("train_dataloader", "val_dataloader", "test_dataloader"):
    d = cfg[name]
    d.batch_size = BATCH_SIZE
    d.num_workers = 2
    d.dataset.data_root = DATASET_DIR + "/"
    d.dataset.type = "YOLOv5CocoDataset"   # reads YOLO txt labels, exposes COCO API
    d.metainfo = dict(classes=CLASSES)

cfg.val_evaluator.data_prefix = ""
cfg.val_evaluator.ann_file = DATASET_DIR + "/labels/val/"
cfg.val_evaluator.metainfo = dict(classes=CLASSES)
cfg.default_hooks = dict(
    timer=dict(type="IterTimerHook"),
    logger=dict(type="LoggerHook", interval=50),
    param_scheduler=dict(type="ParamSchedulerHook"),
    checkpoint=dict(type="CheckpointHook", interval=10, max_keep_ckpts=2),
    sampler_seed=dict(type="DistSamplerSeedHook"),
    visualization=dict(type="DetVisualizationHook"),
)
cfg.param_scheduler = [
    dict(type="LinearLR", start_factor=0.1, by_epoch=False, begin=0, end=500),
    dict(type="CosineLR", T_max=MAX_EPOCHS, by_epoch=True, begin=1, end=MAX_EPOCHS),
]
cfg.optim_wrapper.optimizer.lr = 0.01 * BATCH_SIZE / 8
cfg.train_cfg = dict(type="EpochBasedTrainLoop", max_epochs=MAX_EPOCHS, val_interval=5)
cfg.val_cfg = dict(type="ValLoop")
cfg.test_cfg = dict(type="TestLoop")
cfg.work_dir = WORK_DIR
cfg.default_runtime = dict(log_level="INFO")
cfg.visualizer.vis_backends = [dict(type="LocalVisBackend")]
print("config ready — nano backbone, classes:", len(CLASSES))

In [ ]:
# --- train -----------------------------------------------------------------
from mmengine.runner import Runner
runner = Runner.from_cfg(cfg)
runner.train()

In [ ]:
# --- post-train sanity: detect on a val sheet -------------------------------
import mmcv, torch
from mmdet.apis import init_detector, inference_detector
from PIL import Image, ImageDraw

ckpt = sorted(pathlib.Path(WORK_DIR).rglob("best_coco_bbox_mAP*.pth")) or \
       sorted(pathlib.Path(WORK_DIR).rglob("epoch_*.pth"))
model = init_detector(cfg, str(ckpt[-1]), device="cuda:0" if torch.cuda.is_available() else "cpu")
p = sorted((pathlib.Path(DATASET_DIR) / "images" / "val").glob("*.png"))[0]
res = inference_detector(model, str(p))
im = Image.open(p).convert("RGB"); d = ImageDraw.Draw(im)
n = 0
inst = res.pred_instances
keep = inst.scores > 0.35
for score, b, l in zip(inst.scores[keep], inst.bboxes[keep], inst.labels[keep]):
    x1, y1, x2, y2 = [float(v) for v in b]
    d.rectangle([x1, y1, x2, y2], outline="lime", width=3)
    d.text((x1 + 4, y1 + 4), f"{CLASSES[int(l)]} {float(score):.2f}", fill="lime")
    n += 1
print(f"{n} detections on {p.name}")
im

In [ ]:
# --- export to ONNX for the browser (scan-onnx.ts 'yolox-raw' head) ---------
# Wrapper decodes YOLOX grid outputs -> [1, N, 5+C] rows (x1,y1,x2,y2,obj,cls...)
# in input-pixel coordinates, sigmoid'ed, obj NOT yet multiplied into cls —
# exactly what src/scan-onnx.ts expects for head "yolox-raw".
import torch, torch.nn as nn

model.eval()

class YoloXExport(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m

    def forward(self, x):
        feats = self.model.extract_feat(x)
        outs = self.model.bbox_head(feats)          # [(cls,bbox,obj) per level]
        rows = []
        for lvl, (cls_map, bbox_map, obj_map) in enumerate(outs):
            b, _, h, w = cls_map.shape
            stride = INPUT_SIZE // h
            ys, xs = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
            cx = (xs.float() + 0.5) * stride
            cy = (ys.float() + 0.5) * stride
            cx, cy = cx.reshape(1, 1, -1, 1), cy.reshape(1, 1, -1, 1)
            cls_t = cls_map.flatten(2).transpose(1, 2).unsqueeze(2)   # b,N,1,C
            bbox_t = bbox_map.flatten(2).transpose(1, 2).unsqueeze(2) # b,N,1,4
            obj_t = obj_map.flatten(2).transpose(1, 2).unsqueeze(2)   # b,N,1,1
            l = bbox_t[..., 0:1] * stride; t = bbox_t[..., 1:2] * stride
            r = bbox_t[..., 2:3] * stride; btm = bbox_t[..., 3:4] * stride
            box = torch.cat([cx - l, cy - t, cx + r, cy + btm], dim=-1)
            rows.append(torch.cat([
                box, torch.sigmoid(obj_t).expand(-1, -1, -1, 1), torch.sigmoid(cls_t)
            ], dim=-1))
        out = torch.cat([rr.squeeze(2) for rr in rows], dim=1)        # b,N,5+C
        return out

export_model = YoloXExport(model).cuda() if torch.cuda.is_available() else YoloXExport(model)
dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=next(export_model.parameters()).device)
with torch.no_grad():
    out = export_model(dummy)
print("export head output:", tuple(out.shape), "(expect 1, N, 5 + %d)" % len(CLASSES))
assert out.shape[0] == 1 and out.shape[2] == 5 + len(CLASSES)

torch.onnx.export(
    export_model, dummy, "/kaggle/working/symbols.onnx",
    input_names=["images"], output_names=["output"],
    opset_version=12, do_constant_folding=True,
)
import shutil, json
json.dump({
    "model": "symbols.onnx",
    "input": INPUT_SIZE,
    "classes": CLASSES,
    "score": 0.25,
    "iou": 0.45,
    "head": "yolox-raw",
}, open("/kaggle/working/symbols.json", "w"), indent=2)
print("wrote /kaggle/working/symbols.onnx + symbols.json")

## Ship it

Download `symbols.onnx` + `symbols.json` from the notebook Output panel and drop both into the
repo's `public/models/` directory — that folder is gitignored on purpose (model binaries are
training artifacts, not source). Reload the app and the Scan tool is now the detector:
click any symbol and its whole class gets counted, pre-labelled.

Re-trains don't rebuild anything: replace the two files, reload, done.

**Numbers to watch** (printed during training): `coco/bbox_mAP` on val — synthetic-only
pretraining typically lands mAP@50 in the 0.7–0.9 range for these five clean symbol classes;
real-sheet performance is what the pilot measures. When val mAP plateaus, add FloorPlanCAD
data or more synthetic variety (count/seed) rather than epochs.

**Private data never comes here.** Client tenders are fine-tuned on owned hardware; this
notebook is for the public, Apache-2.0-licensed base model only.